# Part 2 — Upsert & Schema Evolution

This notebook covers `upsert_data()` end-to-end: table auto-creation, all three conflict strategies, automatic schema evolution, composite primary keys, and append mode.

**Prerequisites:** complete [Part 1](Part1_Getting_Started.ipynb) first.

In [ ]:
import pandas as pd
from postgres_connector import PostgresConnector

pg = PostgresConnector(
    host="localhost",
    database="my_db",
    username="postgres",
    password="mysecretpassword",
    schema="tutorial",
)
pg.execute_query("CREATE SCHEMA IF NOT EXISTS tutorial;")

## 1. First Upsert — Auto-Create Table

When the target table does not exist yet, `upsert_data()` creates it automatically by:
1. Inferring PostgreSQL column types from the DataFrame.
2. Running `CREATE TABLE` with the appropriate types.
3. Adding a `PRIMARY KEY` constraint if `primary_key` is supplied.
4. Inserting all rows.

In [ ]:
products = pd.DataFrame({
    "product_id": [1, 2, 3],
    "name":       ["Keyboard", "Mouse", "Monitor"],
    "price":      [89.99, 29.99, 299.99],
    "stock":      [30, 50, 15],
})

pg.upsert_data(
    df=products,
    target_table="products",
    primary_key="product_id",   # creates PK constraint
    conflict_strategy="last",   # default
)

pg.get_data("SELECT * FROM products ORDER BY product_id;")

## 2. Conflict Strategy: `last` (default)

On a primary-key conflict the incoming row **overwrites** all non-key columns.  
Use this for "always keep the latest value" scenarios.

In [ ]:
# product_id 1 price dropped; product_id 4 is brand new
update = pd.DataFrame({
    "product_id": [1, 4],
    "name":       ["Keyboard", "Webcam"],
    "price":      [79.99, 49.99],   # Keyboard price changed
    "stock":      [30, 200],
})

pg.upsert_data(update, "products", primary_key="product_id", conflict_strategy="last")
pg.get_data("SELECT * FROM products ORDER BY product_id;")
# product_id 1 → price is now 79.99 (updated)
# product_id 4 → new row inserted

## 3. Conflict Strategy: `skip`

On a conflict the existing row is **kept unchanged** and the incoming row is discarded.  
Use this when the first write wins — e.g. event deduplication, idempotent imports.

In [ ]:
# Try to overwrite Keyboard with a different name — but skip if already present
attempt = pd.DataFrame({
    "product_id": [1, 5],
    "name":       ["SHOULD BE IGNORED", "Headset"],
    "price":      [0.01, 59.99],
    "stock":      [0, 80],
})

pg.upsert_data(attempt, "products", primary_key="product_id", conflict_strategy="skip")
pg.get_data("SELECT product_id, name, price FROM products ORDER BY product_id;")
# product_id 1 → name still 'Keyboard' (kept); product_id 5 → inserted

## 4. Conflict Strategy: `sum`

On a conflict, **numeric columns are added together** instead of replaced.  
Perfect for inventory restocking, counter accumulation, or budget roll-ups.

In [ ]:
# Current stock: Keyboard=30, Mouse=50.  Receive a new shipment:
shipment = pd.DataFrame({
    "product_id": [1, 2],
    "name":       ["Keyboard", "Mouse"],
    "price":      [79.99, 29.99],  # non-numeric cols use 'last' fallback
    "stock":      [100, 75],       # these get ADDED to existing values
})

pg.upsert_data(shipment, "products", primary_key="product_id", conflict_strategy="sum")
pg.get_data("SELECT product_id, name, stock FROM products WHERE product_id IN (1, 2);")
# Keyboard: 30 + 100 = 130
# Mouse:    50 +  75 = 125

## 5. Automatic Schema Evolution

When your DataFrame gains new columns that do not exist in the DB table yet, `upsert_data()` automatically runs `ALTER TABLE … ADD COLUMN` for each new column before inserting data.

This is controlled by the `auto_evolve_schema` flag (default `True`).

In [ ]:
# Add two completely new columns: 'category' and 'rating'
enriched = pd.DataFrame({
    "product_id": [1, 2, 3],
    "name":       ["Keyboard", "Mouse", "Monitor"],
    "price":      [79.99, 29.99, 299.99],
    "stock":      [130, 125, 15],
    "category":   ["peripherals", "peripherals", "displays"],  # NEW
    "rating":     [4.5, 4.2, 4.8],                             # NEW
})

pg.upsert_data(
    enriched,
    "products",
    primary_key="product_id",
    auto_evolve_schema=True,   # default — ALTER TABLE runs automatically
)

# 'category' and 'rating' columns now exist in the DB
pg.get_data("SELECT product_id, name, category, rating FROM products ORDER BY product_id;")

### Opting out of schema evolution

Set `auto_evolve_schema=False` to **silently drop new columns** from the DataFrame instead of altering the table.  
Use this when the DB schema is locked and unexpected columns should be ignored rather than persisted.

In [ ]:
with_extra_col = pd.DataFrame({
    "product_id":    [1],
    "name":          ["Keyboard"],
    "price":         [79.99],
    "stock":         [130],
    "UNKNOWN_COL":   ["should be dropped"],  # not in DB schema
})

pg.upsert_data(
    with_extra_col,
    "products",
    primary_key="product_id",
    auto_evolve_schema=False,  # UNKNOWN_COL is quietly ignored
)
print("Upsert completed — extra column was dropped before insert.")

## 6. Composite Primary Keys

Pass a **list** to `primary_key` when uniqueness is defined by multiple columns.

In [ ]:
# Each (product_id, warehouse_id) pair is unique
inventory = pd.DataFrame({
    "product_id":   [1, 1, 2],
    "warehouse_id": ["WH-A", "WH-B", "WH-A"],
    "qty":          [50, 80, 120],
})

pg.upsert_data(
    inventory,
    target_table="inventory",
    primary_key=["product_id", "warehouse_id"],  # composite PK
    conflict_strategy="last",
)
pg.get_data("SELECT * FROM inventory ORDER BY product_id, warehouse_id;")

In [ ]:
# Update just WH-A for product 1 — WH-B is untouched
adjustment = pd.DataFrame({
    "product_id":   [1],
    "warehouse_id": ["WH-A"],
    "qty":          [60],      # was 50, now 60
})

pg.upsert_data(adjustment, "inventory", primary_key=["product_id", "warehouse_id"])
pg.get_data("SELECT * FROM inventory ORDER BY product_id, warehouse_id;")

## 7. Append Mode (no primary key)

When `primary_key` is `None` or omitted, every row in the DataFrame is **appended** to the table without any conflict check.  
Useful for append-only event logs, audit trails, or time-series.

In [ ]:
events = pd.DataFrame({
    "event":     ["page_view", "add_to_cart", "checkout"],
    "user_id":   [42, 42, 42],
    "ts":        pd.to_datetime(["2024-01-01 10:00", "2024-01-01 10:05", "2024-01-01 10:12"]),
})

# No primary_key → pure append, no upsert logic
pg.upsert_data(events, target_table="event_log")

# Calling again appends again (duplicate rows are allowed in this mode)
pg.upsert_data(events, target_table="event_log")

count = pg.get_data("SELECT COUNT(*) AS n FROM event_log;")
print(count)   # 6 — 3 rows inserted twice

## 8. Cleanup

In [ ]:
for tbl in ["products", "inventory", "event_log"]:
    pg.execute_query(f'DROP TABLE IF EXISTS {tbl};')

pg.dispose()
print("Done.")

## Summary

| Scenario | Setting |
|----------|---------|
| Always overwrite on conflict | `conflict_strategy="last"` |
| Keep existing on conflict | `conflict_strategy="skip"` |
| Accumulate numbers on conflict | `conflict_strategy="sum"` |
| Add new columns automatically | `auto_evolve_schema=True` (default) |
| Reject new columns silently | `auto_evolve_schema=False` |
| Uniqueness across multiple columns | `primary_key=["col_a", "col_b"]` |
| Pure append, no conflict check | `primary_key=None` (omit) |

**Next:** [Part 3 — Bulk Operations](Part3_Bulk_Operations.ipynb)